# 2. Data Preprocessing Pipeline

**Pipeline overview:**

```
data/raw/          →  data/interim/        →  data/processed/
─────────────────     ──────────────────────  ───────────────────────────────
aio26_train.csv       train_interim.csv        train_processed.csv
aio26_test.csv        test_interim.csv         test_processed.csv
                      fold_id.csv              train_X.csv  /  train_y.csv
                                               test_X.csv
                                               label_encoder.pkl
                                               scaler.pkl
```

**Interim** = cleaned + imputed + missing-flag features (human-readable)  
**Processed** = encoded + log-transformed + scaled (ML-ready)

## 2.1 Imports & Config

In [1]:
import pandas as pd
import numpy as np
import os, pickle, warnings
from sklearn.preprocessing import RobustScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR      = '../data/raw'
INTERIM_DIR  = '../data/interim'
PROC_DIR     = '../data/processed'

os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROC_DIR,    exist_ok=True)

RANDOM_STATE = 42
N_FOLDS      = 5

print('Config ready ✅')
print(f'  Raw      → {RAW_DIR}')
print(f'  Interim  → {INTERIM_DIR}')
print(f'  Processed→ {PROC_DIR}')

Config ready ✅
  Raw      → ../data/raw
  Interim  → ../data/interim
  Processed→ ../data/processed


## 2.2. Load Raw Data

In [2]:
train_raw = pd.read_csv(f'{RAW_DIR}/aio26_train.csv')
test_raw  = pd.read_csv(f'{RAW_DIR}/aio26_test.csv')

print(f'Train raw : {train_raw.shape}')
print(f'Test  raw : {test_raw.shape}')
train_raw.head(3)

Train raw : (12000, 20)
Test  raw : (10000, 19)


,id,Followup_Days,Treatment_Assignment,Patient_Age_Days,Patient_Sex,Ascites_Indicator,Liver_Enlargement,Spider_Angioma,Edema_Status,Bilirubin_Level,Cholesterol_Level,Albumin_Level,Copper_Level,Alkaline_Phosphatase,AST_Level,Triglyceride_Level,Platelet_Count,Prothrombin_Time,Clinical_Stage,Status
0,7523,1416,Treatment_A,26612,Female,Absent,Absent,Absent,Controlled,0.50,NaN,3.517,15.1,726.3,83.97,NaN,258.2,9.87,2.0,C
1,10266,3182,Treatment_A,18647,Female,Absent,Absent,Absent,NaN,0.55,224.8,4.087,30.0,669.6,58.56,65.6,272.0,10.55,3.0,C
2,9705,1372,NaN,23408,Female,NaN,NaN,NaN,NaN,0.82,NaN,3.789,NaN,NaN,NaN,NaN,388.8,9.94,3.0,C


## 2.3 Step 1 → Interim: Clean + Impute + Missing Flags

**Actions:**
- `Patient_Age_Years` = `Patient_Age_Days / 365.25`
- Drop `Edema_Status` (92 % missing) → keep binary flag `Edema_Status_flag`
- Add `is_missing_*` binary flags for all 7 high-missingness features
- **Numerical imputation**: median (computed on train, applied to test)
- **Categorical imputation**: `'Unknown'` category
- Target `Status` label-encoded (train only)

In [3]:
# ── Feature lists ─────────────────────────────────────────────────────────────
HIGH_MISS_NUM  = ['Cholesterol_Level', 'Triglyceride_Level', 'Copper_Level',
                  'Alkaline_Phosphatase', 'AST_Level']
LOW_MISS_NUM   = ['Platelet_Count', 'Prothrombin_Time']
HIGH_MISS_CAT  = ['Treatment_Assignment', 'Ascites_Indicator',
                  'Liver_Enlargement', 'Spider_Angioma']
COMPLETE_NUM   = ['Bilirubin_Level', 'Albumin_Level', 'Patient_Age_Days',
                  'Followup_Days', 'Clinical_Stage']
COMPLETE_CAT   = ['Patient_Sex']

# Features for which we create missing-flag columns
FLAG_FEATURES  = HIGH_MISS_NUM + LOW_MISS_NUM + HIGH_MISS_CAT + ['Edema_Status']


def make_interim(df_in, medians=None, is_train=True):
    """
    Returns (df_interim, medians_dict).
    medians is computed from train and reused for test.
    """
    df = df_in.copy()

    # ── Derived numerical feature ─────────────────────────────────────────────
    df['Patient_Age_Years'] = df['Patient_Age_Days'] / 365.25

    # ── Missing-indicator flags ───────────────────────────────────────────────
    for feat in FLAG_FEATURES:
        if feat in df.columns:
            df[f'{feat}_flag'] = df[feat].isnull().astype(int)

    # ── Drop Edema_Status (keep flag only) ────────────────────────────────────
    if 'Edema_Status' in df.columns:
        df.drop(columns=['Edema_Status'], inplace=True)

    # ── Numerical imputation (median from train) ──────────────────────────────
    all_num_miss = HIGH_MISS_NUM + LOW_MISS_NUM
    if medians is None:                         # compute on train
        medians = {col: df[col].median() for col in all_num_miss if col in df.columns}
    for col, med in medians.items():
        if col in df.columns:
            df[col] = df[col].fillna(med)

    # ── Categorical imputation → 'Unknown' ───────────────────────────────────
    for col in HIGH_MISS_CAT:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    return df, medians


train_interim, train_medians = make_interim(train_raw, is_train=True)
test_interim,  _             = make_interim(test_raw,  medians=train_medians, is_train=False)

print(f'Train interim: {train_interim.shape}')
print(f'Test  interim: {test_interim.shape}')
print(f'Remaining nulls (train): {train_interim.isnull().sum()[train_interim.isnull().sum() > 0].to_dict()}')
train_interim.head(3)

Train interim: (12000, 32)
Test  interim: (10000, 31)
Remaining nulls (train): {}


,id,Followup_Days,Treatment_Assignment,Patient_Age_Days,Patient_Sex,Ascites_Indicator,Liver_Enlargement,Spider_Angioma,Bilirubin_Level,Cholesterol_Level,...,Copper_Level_flag,Alkaline_Phosphatase_flag,AST_Level_flag,Platelet_Count_flag,Prothrombin_Time_flag,Treatment_Assignment_flag,Ascites_Indicator_flag,Liver_Enlargement_flag,Spider_Angioma_flag,Edema_Status_flag
0,7523,1416,Treatment_A,26612,Female,Absent,Absent,Absent,0.50,280.55,...,0,0,0,0,0,0,0,0,0,0
1,10266,3182,Treatment_A,18647,Female,Absent,Absent,Absent,0.55,224.80,...,0,0,0,0,0,0,0,0,0,1
2,9705,1372,Unknown,23408,Female,Unknown,Unknown,Unknown,0.82,280.55,...,1,1,1,0,0,1,1,1,1,1


In [4]:
# ── Stratified K-Fold assignment (train only) ─────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
train_interim['fold_id'] = -1
for fold, (_, val_idx) in enumerate(skf.split(train_interim, train_interim['Status'])):
    train_interim.loc[val_idx, 'fold_id'] = fold

print('Fold distribution:')
print(train_interim.groupby('fold_id')['Status'].value_counts().unstack())

Fold distribution:
Status      C  CL    D
fold_id               
0        1631  63  706
1        1630  63  707
2        1630  63  707
3        1630  64  706
4        1630  64  706


In [ ]:
# ── Save interim ─────────────────────────────────────────────────────────────
train_interim.to_csv(f'{INTERIM_DIR}/train_interim.csv', index=False)
test_interim.to_csv( f'{INTERIM_DIR}/test_interim.csv',  index=False)

# Save fold ids separately for easy loading in modelling notebooks
fold_df = train_interim[['id', 'fold_id']]
fold_df.to_csv(f'{INTERIM_DIR}/fold_id.csv', index=False)

# Save medians so they can be reloaded
pd.Series(train_medians, name='median').to_csv(f'{INTERIM_DIR}/train_medians.csv', header=True)

print('Interim data saved:')
for f in os.listdir(INTERIM_DIR):
    size = os.path.getsize(f'{INTERIM_DIR}/{f}')
    print(f'  {f:35s}  {size/1024:.1f} KB')

✅ Interim data saved:
  fold_id.csv                          96.8 KB
  test_interim.csv                     1568.8 KB
  train_interim.csv                    1929.3 KB
  train_medians.csv                    0.2 KB


## 2.4. Step 2 → Processed: Encode + Log-Transform + Scale

**Actions applied to interim data:**
- `log1p` transform on right-skewed features (skew > 1.5)
- `OrdinalEncoder` for categorical features
- `RobustScaler` on all numerical features (robust to outliers)
- `LabelEncoder` for target `Status`
- Scaler & encoder objects persisted as `.pkl` files

In [6]:
# ── Column groups for processed stage ────────────────────────────────────────
LOG_TRANSFORM_COLS = [
    'Bilirubin_Level', 'Cholesterol_Level', 'Copper_Level',
    'Alkaline_Phosphatase', 'AST_Level', 'Triglyceride_Level'
]

CAT_COLS = [
    'Treatment_Assignment', 'Patient_Sex',
    'Ascites_Indicator', 'Liver_Enlargement', 'Spider_Angioma'
]

TARGET_COL = 'Status'
DROP_COLS  = ['id', 'fold_id']  # not features for model


def make_processed(df_interim, cat_encoder=None, scaler=None,
                   label_enc=None, is_train=True):
    """
    Returns processed DataFrame + fitted transformers (train only).
    """
    df = df_interim.copy()

    # ── Feature: Bilirubin / Albumin ratio ────────────────────────────────────
    df['Bilirubin_Albumin_Ratio'] = df['Bilirubin_Level'] / (df['Albumin_Level'] + 1e-6)

    # ── log1p transform ───────────────────────────────────────────────────────
    for col in LOG_TRANSFORM_COLS:
        if col in df.columns:
            df[col] = np.log1p(df[col])
    # also log1p the ratio
    df['Bilirubin_Albumin_Ratio'] = np.log1p(df['Bilirubin_Albumin_Ratio'])

    # ── Categorical encoding ──────────────────────────────────────────────────
    cats_present = [c for c in CAT_COLS if c in df.columns]
    if cat_encoder is None:                         # fit on train
        cat_encoder = OrdinalEncoder(
            handle_unknown='use_encoded_value', unknown_value=-1
        )
        df[cats_present] = cat_encoder.fit_transform(df[cats_present].astype(str))
    else:
        df[cats_present] = cat_encoder.transform(df[cats_present].astype(str))

    # ── Identify all numerical feature columns to scale ───────────────────────
    meta_cols   = DROP_COLS + ([TARGET_COL] if is_train else [])
    num_cols    = [c for c in df.columns
                   if c not in meta_cols + cats_present
                   and df[c].dtype != object]

    # ── RobustScaler ─────────────────────────────────────────────────────────
    if scaler is None:                              # fit on train
        scaler = RobustScaler()
        df[num_cols] = scaler.fit_transform(df[num_cols])
    else:
        df[num_cols] = scaler.transform(df[num_cols])

    # ── Target encoding (train only) ──────────────────────────────────────────
    if is_train and TARGET_COL in df.columns:
        if label_enc is None:
            label_enc = LabelEncoder()
            df[TARGET_COL] = label_enc.fit_transform(df[TARGET_COL])
        else:
            df[TARGET_COL] = label_enc.transform(df[TARGET_COL])

    return df, cat_encoder, scaler, label_enc


train_proc, cat_enc, scaler, le = make_processed(train_interim, is_train=True)
test_proc,  _,      _,      _  = make_processed(test_interim,
                                                  cat_encoder=cat_enc,
                                                  scaler=scaler,
                                                  is_train=False)

print(f'Train processed : {train_proc.shape}')
print(f'Test  processed : {test_proc.shape}')
print(f'Target classes  : {dict(zip(le.classes_, le.transform(le.classes_)))}')
train_proc.head(3)

Train processed : (12000, 34)
Test  processed : (10000, 32)
Target classes  : {'C': 0, 'CL': 1, 'D': 2}


,id,Followup_Days,Treatment_Assignment,Patient_Age_Days,Patient_Sex,Ascites_Indicator,Liver_Enlargement,Spider_Angioma,Bilirubin_Level,Cholesterol_Level,...,AST_Level_flag,Platelet_Count_flag,Prothrombin_Time_flag,Treatment_Assignment_flag,Ascites_Indicator_flag,Liver_Enlargement_flag,Spider_Angioma_flag,Edema_Status_flag,fold_id,Bilirubin_Albumin_Ratio
0,7523,-0.239004,1.0,1.199607,0.0,0.0,0.0,0.0,-0.434087,0.00000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,4,-0.357736
1,10266,0.903299,1.0,-0.161583,0.0,0.0,0.0,0.0,-0.372499,-0.22066,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,-0.383954
2,9705,-0.267464,3.0,0.652055,0.0,2.0,2.0,2.0,-0.070886,0.00000,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,4,-0.110196


In [7]:
# ── Verify no remaining nulls ─────────────────────────────────────────────────
nulls_train = train_proc.isnull().sum().sum()
nulls_test  = test_proc.isnull().sum().sum()
print(f'Nulls in train_proc : {nulls_train}')
print(f'Nulls in test_proc  : {nulls_test}')
assert nulls_train == 0, '❌ Train processed has NaN values!'
assert nulls_test  == 0, '❌ Test processed has NaN values!'
print('✅ No missing values in processed data')

Nulls in train_proc : 0
Nulls in test_proc  : 0
✅ No missing values in processed data


## 2.5. Save Processed Data

In [ ]:
# ── Separate X / y / meta ─────────────────────────────────────────────────────
feature_cols = [c for c in train_proc.columns if c not in ['id', 'fold_id', TARGET_COL]]

train_X = train_proc[feature_cols]
train_y = train_proc[[TARGET_COL]]
test_X  = test_proc[[c for c in feature_cols if c in test_proc.columns]]

# ── Save processed CSVs ───────────────────────────────────────────────────────
train_proc.to_csv( f'{PROC_DIR}/train_processed.csv', index=False)
test_proc.to_csv(  f'{PROC_DIR}/test_processed.csv',  index=False)
train_X.to_csv(    f'{PROC_DIR}/train_X.csv',          index=False)
train_y.to_csv(    f'{PROC_DIR}/train_y.csv',          index=False)
test_X.to_csv(     f'{PROC_DIR}/test_X.csv',           index=False)

# ── Persist transformers ──────────────────────────────────────────────────────
with open(f'{PROC_DIR}/label_encoder.pkl',  'wb') as f: pickle.dump(le,      f)
with open(f'{PROC_DIR}/cat_encoder.pkl',    'wb') as f: pickle.dump(cat_enc, f)
with open(f'{PROC_DIR}/scaler.pkl',         'wb') as f: pickle.dump(scaler,  f)

# ── Also save feature list for modelling notebooks ────────────────────────────
pd.Series(feature_cols, name='feature').to_csv(f'{PROC_DIR}/feature_list.csv', index=False)

print('Processed data saved:')
for f in sorted(os.listdir(PROC_DIR)):
    size = os.path.getsize(f'{PROC_DIR}/{f}')
    print(f'  {f:35s}  {size/1024:.1f} KB')

✅ Processed data saved:
  cat_encoder.pkl                      0.9 KB
  feature_list.csv                     0.6 KB
  label_encoder.pkl                    0.2 KB
  scaler.pkl                           1.4 KB
  test_X.csv                           2845.6 KB
  test_processed.csv                   2904.2 KB
  train_X.csv                          3416.2 KB
  train_processed.csv                  3524.7 KB
  train_y.csv                          35.2 KB


## 2.6. Final Verification

In [9]:
print('═' * 60)
print('DATA PIPELINE SUMMARY')
print('═' * 60)

# Raw
print(f'\n📂 data/raw/')
for f in sorted(os.listdir('../data/raw')):
    size = os.path.getsize(f'../data/raw/{f}')
    print(f'  {f:40s}  {size/1024:.0f} KB')

# Interim
print(f'\n📂 data/interim/')
for f in sorted(os.listdir(INTERIM_DIR)):
    size = os.path.getsize(f'{INTERIM_DIR}/{f}')
    print(f'  {f:40s}  {size/1024:.0f} KB')

# Processed
print(f'\n📂 data/processed/')
for f in sorted(os.listdir(PROC_DIR)):
    size = os.path.getsize(f'{PROC_DIR}/{f}')
    print(f'  {f:40s}  {size/1024:.0f} KB')

print(f'\nFeature count in processed  : {len(feature_cols)}')
print(f'Train X shape               : {train_X.shape}')
print(f'Test  X shape               : {test_X.shape}')
print(f'Label mapping               : {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'Folds                       : {N_FOLDS}-fold stratified CV')
print('\n✅ Pipeline complete! Ready for modelling.')

════════════════════════════════════════════════════════════
DATA PIPELINE SUMMARY
════════════════════════════════════════════════════════════

📂 data/raw/
  aio26_sample-submission.csv               332 KB
  aio26_test.csv                            934 KB
  aio26_train.csv                           1135 KB

📂 data/interim/
  fold_id.csv                               97 KB
  test_interim.csv                          1569 KB
  train_interim.csv                         1929 KB
  train_medians.csv                         0 KB

📂 data/processed/
  cat_encoder.pkl                           1 KB
  feature_list.csv                          1 KB
  label_encoder.pkl                         0 KB
  scaler.pkl                                1 KB
  test_X.csv                                2846 KB
  test_processed.csv                        2904 KB
  train_X.csv                               3416 KB
  train_processed.csv                       3525 KB
  train_y.csv                               35

---
## Pipeline Recap

| Stage | Location | Contents |
|---|---|---|
| **Raw** | `data/raw/` | Original untouched CSVs |
| **Interim** | `data/interim/` | Cleaned + imputed + flags + fold IDs |
| **Processed** | `data/processed/` | Encoded + log-transformed + scaled; X/y split; `.pkl` transformers |

**New features added:**
- `Patient_Age_Years`: age in human-readable units
- `{feature}_flag`: binary missing indicator for 10 features
- `Bilirubin_Albumin_Ratio`: liver synthetic function proxy